Chapter 3 Exercise solutions


In [ ]:
from importlib.metadata import version  # 从标准库导入 version 函数，用于查询已安装包的版本号

import torch  # 导入 PyTorch，本章后续的张量与自注意力计算都依赖它
print("torch version:", version("torch"))  # 打印当前环境安装的 torch 版本，便于确认依赖是否满足

Exercise 3.1

In [ ]:
# 构造一个形状为 (6, 3) 的输入张量：
# 6 表示序列长度（6 个 token，对应 "Your journey starts with one step" 这句话的 6 个词）
# 3 表示每个 token 的嵌入维度 d_in
# 每一行是一个 token 的嵌入向量，行尾的英文注释标注了该 token 对应原句中的第几个词 (x^i)
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

d_in, d_out = 3, 2  # d_in：输入嵌入维度（与 inputs 的列数一致，为 3）；d_out：注意力输出的维度（Q/K/V 投影后的维度，为 2）

In [ ]:
import torch.nn as nn  # 导入 nn 模块，用于定义可训练参数与模型层

class SelfAttention_v1(nn.Module):
    # 第一版自注意力实现：直接用 nn.Parameter 手写 Wq/Wk/Wv 权重矩阵（不使用 nn.Linear）

    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        # 三个可训练权重矩阵，形状均为 (d_in, d_out)，初始值为 [0,1) 均匀分布随机数
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # x 形状: (num_tokens, d_in)
        # 通过矩阵乘法把每个 token 的嵌入投影到 key/query/value 空间，输出形状均为 (num_tokens, d_out)
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        # 注意力打分（未缩放/未归一化的原始分数，书中称为 omega）
        # queries @ keys.T 形状: (num_tokens, num_tokens)，第 i 行第 j 列表示 token i 对 token j 的注意力得分
        attn_scores = queries @ keys.T # omega
        # 缩放点积注意力：除以 sqrt(d_k)（keys 的最后一维，即 d_out）以稳定梯度，
        # 再对最后一维（每一行，即针对每个 query 的所有 key）做 softmax，使每行权重之和为 1
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        # 用注意力权重对 values 做加权求和，得到每个 token 的上下文向量，形状: (num_tokens, d_out)
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)  # 固定随机种子，保证权重初始化可复现
sa_v1 = SelfAttention_v1(d_in, d_out)  # 实例化 v1 版本的自注意力模块

In [ ]:
class SelfAttention_v2(nn.Module):
    # 第二版自注意力实现：用 nn.Linear(bias=False) 代替手写的 nn.Parameter 矩阵，
    # 效果等价但参数初始化方式不同（Linear 默认使用 kaiming_uniform 初始化），且支持批量高效计算

    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_out = d_out
        # 三个线性层，权重形状为 (d_out, d_in)（注意 nn.Linear 的权重存储顺序与手写矩阵相反）
        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        # x 形状: (num_tokens, d_in)；Linear 内部等价于 x @ weight.T，输出形状: (num_tokens, d_out)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # 注意力打分，形状: (num_tokens, num_tokens)
        attn_scores = queries @ keys.T
        # 缩放点积注意力，dim=1 对二维张量而言等价于 dim=-1，同样是对每一行做归一化
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=1)

        # 加权求和得到上下文向量，形状: (num_tokens, d_out)
        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)  # 固定随机种子，保证权重初始化可复现
sa_v2 = SelfAttention_v2(d_in, d_out)  # 实例化 v2 版本的自注意力模块

In [ ]:
# 练习 3.1：让 v1 与 v2 使用完全相同的权重，从而验证两种实现在数学上等价
# nn.Linear.weight 的形状是 (d_out, d_in)，而 v1 中手写的 W_query/W_key/W_value 形状是 (d_in, d_out)，
# 因此需要用 .T 转置后再赋值给 v1 对应的 nn.Parameter
sa_v1.W_query = torch.nn.Parameter(sa_v2.W_query.weight.T)
sa_v1.W_key = torch.nn.Parameter(sa_v2.W_key.weight.T)
sa_v1.W_value = torch.nn.Parameter(sa_v2.W_value.weight.T)

In [ ]:
sa_v1(inputs)  # 用同步权重后的 v1 模块对 inputs 做前向计算，输出形状: (6, 2)（6 个 token，每个输出 2 维上下文向量）

In [ ]:
sa_v2(inputs)  # 用 v2 模块对相同的 inputs 做前向计算；权重已与 v1 对齐，结果应与上一单元输出一致